Now that I understood what are the things inside the data, I managed to create three data frames. 

In the challenge they said **The best solutions will likely come from clever feature engineering that can extract the most predictive signals from both the initial matchup and the flow of the early battle.**

So I tried to make two things:

1. A dataframe of initial matchup - ie, some of the details of all 6 pokemon of p1 and lead pokemon of p2(we can extract more info by using the same code.):

   - Along with that I created another dataframe of P1 lead pokemon vs P2 lead pokemon, to check whether the first pokemon of P1 have any advantage disadvantage.
   - We can also check the item matchup between P1 lead vs P2 lead pokemon.
   - Also I want to discuss; rather than these raw data, we could have something like total stat strength of P1, to get an idea whether the pokemon team he put in each battle is whether aggressive or defensive. 
    
    
    
3. Battle flow : This is the turn by turn snapshot of what is happening, we can check if the hp is decreasing or increasing, whether P2 put negative effects on P1, final state of the pokemon at the 30th turn etc.

So you can comment edit this notebook! put comments or what you think by making a markdown cell.

In [1]:
import json
import pandas as pd
import numpy as np


# ==========================================
# 1. LOAD THE TRAINING DATA
# ==========================================
train_file_path = 'data/train.jsonl'
train_data = []

with open(train_file_path, 'r') as f:
    for line in f:
        train_data.append(json.loads(line))

print(f"Successfully loaded {len(train_data)} training battles.")



# ==========================================
# 2. STATIC MATCH UP
# ==========================================

initial_matchup = []

lead_matchup = []

all_turns = []

player_win_counter = 0

for battle in train_data:

    player_won = battle.get('player_won', None)




for battle in train_data:
    
    battle_id = battle.get('battle_id')

    player_won = battle.get('player_won', None)
    
    # counting the number of times the player 1 - won the battle
    if player_won == 1:
        player_win_counter+=1

        
    # ==========================================
    # 2. INITIAL MATCH UP 
    # ==========================================

    # Player 1 details
    p1_team = battle.get('p1_team_details',[])

    # Player 1 Lead details
    p1_lead = p1_team[0] if len(p1_team) > 0 else {}
    p1_lead_name = p1_lead.get('name', 'Unknown')
    p1_lead_type = p1_lead.get('types', [])
    p1_lead_item = p1_lead.get('item', 'NaN')

    # This is all the Player 1 pokemon details - you can extract stats, level etc with same as below 
    # then with the Dataframe we can take mean, max, difference etc.
    p1_names = [pok.get('name', 'Unknown') for pok in p1_team]
    p1_types = [pok.get('types', []) for pok in p1_team]
    p1_items = [pok.get('item', 'NaN') for pok in p1_team]

    # Player 2 lead details - No item in the player 2
    p2_lead = battle.get('p2_lead_details', {}) or {}
    p2_lead_name = p2_lead.get('name', 'Unknown')
    p2_lead_type = p2_lead.get('types', [])


    initial_matchup.append({
        'battle_id': battle_id,
        'p1_names':p1_names,
        'p1_types':p1_types,
        'p1_items':p1_items,
        'p2_lead_name': p2_lead_name,
        'p2_lead_type': p2_lead_type,
        'player_won': player_won


        
    })
   

    lead_matchup.append({
        'battle_id': battle_id,
        'p1_lead_name': p1_lead_name,
        'p1_lead_type': p1_lead_type,
        'p1_lead_item': p1_lead_item,
        'p2_lead_name': p2_lead_name,
        'p2_lead_type': p2_lead_type,
        'player_won': player_won
    })



    # ===================================================
    # 3. THIS IS THE FLOW OF THE EARLY BATTLE - 30 TURN
    # ===================================================

    timeline = battle.get('battle_timeline', [])

    
    for turn_data in timeline:
        turn_number = turn_data.get('turn')

        #pokemon state details
        p1_state = turn_data.get('p1_pokemon_state', {})
        p1_name = p1_state.get('name', 'Unknown')
        p1_hp = p1_state.get('hp_pct', 0.0)
        p1_status = p1_state.get('status', 'Unknown')
        p1_effects = p1_state.get('effects',[])

        #move details
        p1_move = turn_data.get('p1_move_details',{}) or {}
        p1_type = p1_move.get('type', 'NaN')
        p1_category = p1_move.get('category', 'NaN')
        
        

        
        p2_state = turn_data.get('p2_pokemon_state', {})
        p2_name = p2_state.get('name', 'Unknown')
        p2_hp = p2_state.get('hp_pct', 0.0)
        p2_status = p2_state.get('status', 'Unknown')
        p2_effects = p2_state.get('effects',[])

        p2_move = turn_data.get('p2_move_details',{}) or {}
        p2_type = p2_move.get('type', 'NaN')
        p2_category = p2_move.get('category', 'NaN')

        
        
        # including the battle outcome
        all_turns.append({
            'battle_id': battle_id,
            'player_won': player_won, 
            'turn': turn_number,
            'p1_active_pokemon': p1_name,
            'p1_hp_pct': p1_hp,
            'p1_status': p1_status,
            'p1_effects': p1_effects,
            'p1_move_type': p1_type,
            'p1_move_category': p1_category,
            'p2_active_pokemon': p2_name,
            'p2_hp_pct': p2_hp,
            'p2_status': p2_status,
            'p2_effects': p2_effects,
            'p2_move_type': p2_type,
            'p2_move_category': p2_category,
            
        })
        

# Convert to three DataFrames - P1 lead vs P2 Lead, intital pokemon details and battle flow
df_lead_matchup = pd.DataFrame(lead_matchup)
df_initial_matchup = pd.DataFrame(initial_matchup)
df_battleflow = pd.DataFrame(all_turns)

print("\n","="*20, "P1 lead vs P2 lead", "="*20,"\n")
display(df_lead_matchup.head(5))

print("\n","="*20, "P1 all pokemon vs P2 lead", "="*20,"\n")
display(df_initial_matchup.head(5))

print("\n","="*20, "Turn by Turn Battle", "="*20,"\n")
display(df_battleflow.head(30))



#display(df_snapshots.iloc[30:60])

# calculating the win percentage by normal method
player_winrate = player_win_counter/len(train_data)
print(f"Player 1 Win Rate :{player_winrate * 100} % of the time")

# calculating the win percentage by pandas oneliner
print(f"Player 1 Win Rate: {df_initial_matchup['player_won'].mean() * 100:.2f}%")


Successfully loaded 10000 training battles.

 ==================== P1 lead vs P2 lead ==================== 



,battle_id,p1_lead_name,p1_lead_type,p1_lead_item,p2_lead_name,p2_lead_type,player_won
0,0,tyranitar,"[dark, rock]",leftovers,forretress,"[bug, steel]",0
1,1,raikou,"[electric, notype]",leftovers,raikou,"[electric, notype]",0
2,2,raikou,"[electric, notype]",leftovers,exeggutor,"[grass, psychic]",0
3,3,zapdos,"[electric, flying]",leftovers,cloyster,"[ice, water]",0
4,4,nidoking,"[ground, poison]",leftovers,zapdos,"[electric, flying]",0



 ==================== P1 all pokemon vs P2 lead ==================== 



,battle_id,p1_names,p1_types,p1_items,p2_lead_name,p2_lead_type,player_won
0,0,"[tyranitar, typhlosion, golem, umbreon, alakaz...","[[dark, rock], [fire, notype], [ground, rock],...","[leftovers, mintberry, berserkgene, leftovers,...",forretress,"[bug, steel]",0
1,1,"[raikou, skarmory, forretress, misdreavus, sno...","[[electric, notype], [flying, steel], [bug, st...","[leftovers, leftovers, leftovers, leftovers, l...",raikou,"[electric, notype]",0
2,2,"[raikou, heracross, tyranitar, cloyster, snorl...","[[electric, notype], [bug, fighting], [dark, r...","[leftovers, leftovers, leftovers, leftovers, l...",exeggutor,"[grass, psychic]",0
3,3,"[zapdos, snorlax, cloyster, skarmory, blissey,...","[[electric, flying], [normal, notype], [ice, w...","[leftovers, leftovers, leftovers, leftovers, l...",cloyster,"[ice, water]",0
4,4,"[nidoking, exeggutor, steelix, zapdos, snorlax...","[[ground, poison], [grass, psychic], [ground, ...","[leftovers, leftovers, leftovers, leftovers, l...",zapdos,"[electric, flying]",0



 ==================== Turn by Turn Battle ==================== 



,battle_id,player_won,turn,p1_active_pokemon,p1_hp_pct,p1_status,p1_effects,p1_move_type,p1_move_category,p2_active_pokemon,p2_hp_pct,p2_status,p2_effects,p2_move_type,p2_move_category
0,0,0,1,tyranitar,1.00,nostatus,[noeffect],ground,physical,forretress,0.86,nostatus,[noeffect],poison,status
1,0,0,2,typhlosion,1.00,tox,[noeffect],NaN,NaN,forretress,0.93,nostatus,[noeffect],poison,status
2,0,0,3,typhlosion,0.94,tox,[noeffect],fire,special,zapdos,0.61,nostatus,[noeffect],NaN,NaN
3,0,0,4,typhlosion,0.03,tox,[noeffect],fire,special,zapdos,0.26,nostatus,[noeffect],normal,physical
4,0,0,5,typhlosion,1.00,nostatus,[noeffect],psychic,status,snorlax,1.00,nostatus,[noeffect],NaN,NaN
5,0,0,6,typhlosion,1.00,nostatus,[noeffect],electric,special,snorlax,0.94,nostatus,[noeffect],threequestionmarks,status
6,0,0,7,typhlosion,1.00,nostatus,[noeffect],fire,special,snorlax,0.75,nostatus,[noeffect],threequestionmarks,status
7,0,0,8,typhlosion,0.02,nostatus,[noeffect],fire,special,snorlax,0.36,nostatus,[noeffect],normal,physical
8,0,0,9,typhlosion,0.02,nostatus,[noeffect],fire,special,snorlax,1.00,slp,[noeffect],psychic,status
9,0,0,10,typhlosion,1.00,slp,[noeffect],psychic,status,snorlax,1.00,slp,[noeffect],normal,status


Player 1 Win Rate :50.0 % of the time
Player 1 Win Rate: 50.00%


In [45]:
# Using pandas to load the train and test
import json
import pandas as pd
from pathlib import Path

train_file_path = 'data/train.jsonl'
test_file_path = 'data/test.jsonl'


df_train = pd.read_json(train_file_path, lines=True)

df_test = pd.read_json(test_file_path, lines = True)


display(df_train.head(5))
display(df_test.head(5))





,battle_id,p1_team_details,p2_lead_details,battle_timeline,player_won
0,0,"[{'name': 'tyranitar', 'level': 100, 'types': ...","{'name': 'forretress', 'level': 100, 'types': ...","[{'turn': 1, 'p1_pokemon_state': {'name': 'tyr...",0
1,1,"[{'name': 'raikou', 'level': 100, 'types': ['e...","{'name': 'raikou', 'level': 100, 'types': ['el...","[{'turn': 1, 'p1_pokemon_state': {'name': 'rai...",0
2,2,"[{'name': 'raikou', 'level': 100, 'types': ['e...","{'name': 'exeggutor', 'level': 100, 'types': [...","[{'turn': 1, 'p1_pokemon_state': {'name': 'her...",0
3,3,"[{'name': 'zapdos', 'level': 100, 'types': ['e...","{'name': 'cloyster', 'level': 100, 'types': ['...","[{'turn': 1, 'p1_pokemon_state': {'name': 'zap...",0
4,4,"[{'name': 'nidoking', 'level': 100, 'types': [...","{'name': 'zapdos', 'level': 100, 'types': ['el...","[{'turn': 1, 'p1_pokemon_state': {'name': 'nid...",0


,battle_id,p1_team_details,p2_lead_details,battle_timeline
0,0,"[{'name': 'raikou', 'level': 100, 'types': ['e...","{'name': 'raikou', 'level': 100, 'types': ['el...","[{'turn': 1, 'p1_pokemon_state': {'name': 'gol..."
1,1,"[{'name': 'snorlax', 'level': 100, 'types': ['...","{'name': 'snorlax', 'level': 100, 'types': ['n...","[{'turn': 1, 'p1_pokemon_state': {'name': 'rhy..."
2,2,"[{'name': 'snorlax', 'level': 100, 'types': ['...","{'name': 'nidoking', 'level': 100, 'types': ['...","[{'turn': 1, 'p1_pokemon_state': {'name': 'sno..."
3,3,"[{'name': 'raikou', 'level': 100, 'types': ['e...","{'name': 'tyranitar', 'level': 100, 'types': [...","[{'turn': 1, 'p1_pokemon_state': {'name': 'rai..."
4,4,"[{'name': 'electrode', 'level': 100, 'types': ...","{'name': 'qwilfish', 'level': 100, 'types': ['...","[{'turn': 1, 'p1_pokemon_state': {'name': 'ele..."


In [46]:

# p1_team_details is a list of dictionaries, so I use explode() function 
# to iterate over the list of dict and make rows
df_explode_p1team = df_train.explode('p1_team_details').reset_index(drop=True)

# expected output is 6 pokemon per battle for 10000 battle = 60000
df_explode_p1team.info() 


print("-"*100)

# The new rows contains dictionaries and I make coloumns for each dict keys
df_flat_p1team = pd.json_normalize(df_explode_p1team['p1_team_details'])
df_flat_p1team.info()


<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   battle_id        60000 non-null  int64 
 1   p1_team_details  60000 non-null  object
 2   p2_lead_details  60000 non-null  object
 3   battle_timeline  60000 non-null  object
 4   player_won       60000 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 2.3+ MB
----------------------------------------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      60000 non-null  str   
 1   level     60000 non-null  int64 
 2   types     60000 non-null  object
 3   base_hp   60000 non-null  int64 
 4   base_atk  60000 non-null  int64 
 5   base_def  60000 non-null  int64 
 6   base_spa  60000 non-null  int6

In [47]:

# p2_lead_details is a single dictionary, so no need to explode

df_flat_p2team = pd.json_normalize(df_train['p2_lead_details'])

# expected output is 1 lead pokemon per battle for 10000 battle = 10000
df_flat_p2team.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      10000 non-null  str   
 1   level     10000 non-null  int64 
 2   types     10000 non-null  object
 3   base_hp   10000 non-null  int64 
 4   base_atk  10000 non-null  int64 
 5   base_def  10000 non-null  int64 
 6   base_spa  10000 non-null  int64 
 7   base_spd  10000 non-null  int64 
 8   base_spe  10000 non-null  int64 
dtypes: int64(7), object(1), str(1)
memory usage: 703.3+ KB


In [48]:

df_explode_battle_timeline = df_train.explode('battle_timeline').reset_index(drop=True)

# expected output is 30 turn per battle_timeline with a total 10000 battle = 300000
df_explode_battle_timeline.info()

print("-"*100)

df_flat_battle_timeline = pd.json_normalize(df_explode_battle_timeline['battle_timeline'])

df_flat_battle_timeline.info()



<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 5 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   battle_id        300000 non-null  int64 
 1   p1_team_details  300000 non-null  object
 2   p2_lead_details  300000 non-null  object
 3   battle_timeline  300000 non-null  object
 4   player_won       300000 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.4+ MB
----------------------------------------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 33 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   turn                         300000 non-null  int64  
 1   p1_pokemon_state.name        300000 non-null  str    
 2   p1_pokemon_state.hp_pct      300000 non-null  float64
 3   p1_pokemon_state.status    

**With this I identified some variables type as objects and I will try to further inspect 
that and try to print out their contents. 
My aim is to make categorical values that give us some more info about the battles and 
there by improving our features, training and prediction**

In [49]:

# Loop through each column in the dataframe
for column_name in df_flat_p1team.columns:
    
    # Check if the data type is string (object)
    if df_flat_p1team[column_name].dtype == 'object':
        
        print(f"Looking at column: {column_name}")
        
        # Create a variable for the values
        column_data = df_flat_p1team[column_name].astype(str)
        
        # Count the values
        top_values = column_data.value_counts()
        print(top_values.tail())
        print("-" * 20)



Looking at column: types
types
['dark', 'flying']       22
['dragon', 'notype']     20
['flying', 'psychic']    20
['fire', 'rock']         10
['bug', 'grass']         10
Name: count, dtype: int64
--------------------


For player 1 we can list out all the pokemons, their types and items used for the battle

In [50]:

# print(f"Name of the Poekmons = \n{df_flat_p1team['name'].unique()}\n")

print(f"Pokemon Types = \n{df_flat_p1team['types'].explode().unique()}\n")

print(f"Items =\n {df_flat_p1team['item'].unique()}")



Pokemon Types = 
<StringArray>
[    'dark',     'rock',     'fire',   'notype',   'ground',  'psychic',
 'electric',   'flying',    'steel',      'bug',    'ghost',   'normal',
    'water', 'fighting',      'ice',   'poison',    'grass',   'dragon']
Length: 18, dtype: str

Items =
 <StringArray>
[   'leftovers',    'mintberry',  'berserkgene', 'miracleberry',
    'thickclub',    'goldberry',     'charcoal',    'lightball',
   'berryjuice',        'berry',    'quickclaw', 'przcureberry',
    'oranberry',    'hardstone',     'lumberry',    'kingsrock',
     'spelltag',        'stick',    'scopelens',   'poisonbarb',
    'focusband',       'magnet',       'noitem',     'softsand',
      'pinkbow',     'pokeball',  'miracleseed',  'sitrusberry',
 'brightpowder', 'silverpowder',    'blackbelt',    'sharpbeak',
  'metalpowder',  'mysticwater',  'dragonscale',  'chestoberry',
 'nevermeltice',    'metalcoat', 'mysteryberry', 'twistedspoon',
  'polkadotbow', 'blackglasses',  'bitterberry',    '

For Player 2 we have only one information, which is the first pokemon it is leading with and it's type

In [51]:

# Loop through each column in the dataframe
for column_name in df_flat_p2team.columns:
    
    # Check if the data type is string (object)
    if df_flat_p2team[column_name].dtype == 'object':
        
        print(f"Looking at column: {column_name}")
        
        # Create a variable for the values
        column_data = df_flat_p2team[column_name].astype(str)
        
        # Count the values
        top_values = column_data.value_counts()
        print(top_values.tail())
        print("-" * 20)


Looking at column: types
types
['rock', 'water']        2
['bug', 'rock']          2
['flying', 'psychic']    1
['flying', 'ground']     1
['dark', 'ice']          1
Name: count, dtype: int64
--------------------


In [52]:
'''
print(f"Name of the Poekmons = \n{df_flat_p2team['name'].unique()}\n")

print(f"Pokemon Types = \n{df_flat_p2team['types'].explode().unique()}\n")
'''

'\nprint(f"Name of the Poekmons = \n{df_flat_p2team[\'name\'].unique()}\n")\n\nprint(f"Pokemon Types = \n{df_flat_p2team[\'types\'].explode().unique()}\n")\n'

Now we go through the Battle Timeline and extract some information which is relevant such as the status of pokemons during the battle like paralysis, fire, freeze, poison. 

??may be we can use them to give weightage to the chance of winning or loosing

In [53]:
'''

# Loop through each column in the dataframe
for column_name in df_flat_battle_timeline.columns:
    
    # Check if the data type is string (object)
    if df_flat_battle_timeline[column_name].dtype == 'object':
        
        print(f"Looking at column: {column_name}")
        
        # Create a variable for the values
        column_data = df_flat_battle_timeline[column_name].astype(str)
        
        # Count the values
        top_values = column_data.value_counts()
        print(top_values.tail())
        print("-" * 20)

'''

'\n\n# Loop through each column in the dataframe\nfor column_name in df_flat_battle_timeline.columns:\n\n    # Check if the data type is string (object)\n    if df_flat_battle_timeline[column_name].dtype == \'object\':\n\n        print(f"Looking at column: {column_name}")\n\n        # Create a variable for the values\n        column_data = df_flat_battle_timeline[column_name].astype(str)\n\n        # Count the values\n        top_values = column_data.value_counts()\n        print(top_values.tail())\n        print("-" * 20)\n\n'

Now we list all the status pokemon gone through in the battle timeline

In [54]:
'''

# Select both P1 and P2 Pokemon Status, then explode them to break the lists apart
p1_status = df_flat_battle_timeline['p1_pokemon_state.status'].explode()
p2_status = df_flat_battle_timeline['p2_pokemon_state.status'].explode()

# Combine and find the unique values
all_status = pd.concat([p1_status, p2_status]).unique()

print(all_status)

'''

"\n\n# Select both P1 and P2 Pokemon Status, then explode them to break the lists apart\np1_status = df_flat_battle_timeline['p1_pokemon_state.status'].explode()\np2_status = df_flat_battle_timeline['p2_pokemon_state.status'].explode()\n\n# Combine and find the unique values\nall_status = pd.concat([p1_status, p2_status]).unique()\n\nprint(all_status)\n\n"

Another important one - which is the sepcial effects used by the pokemons

In [55]:
'''

# Select both P1 and P2 effects, then explode them to break the lists apart
p1_effects = df_flat_battle_timeline['p1_pokemon_state.effects'].explode()
p2_effects = df_flat_battle_timeline['p2_pokemon_state.effects'].explode()

# Combine and find the unique values
all_effects = pd.concat([p1_effects, p2_effects]).unique()

print(all_effects)
#type(all_effects)

'''

"\n\n# Select both P1 and P2 effects, then explode them to break the lists apart\np1_effects = df_flat_battle_timeline['p1_pokemon_state.effects'].explode()\np2_effects = df_flat_battle_timeline['p2_pokemon_state.effects'].explode()\n\n# Combine and find the unique values\nall_effects = pd.concat([p1_effects, p2_effects]).unique()\n\nprint(all_effects)\n#type(all_effects)\n\n"

This tells us the elemental move. 

Why important? example: water VS fire, fire takes damage

In [56]:
'''

# Select both P1 and P2 move types, then explode them to break the lists apart
p1_move_type = df_flat_battle_timeline['p1_move_details.type'].explode()
p2_move_type = df_flat_battle_timeline['p2_move_details.type'].explode()

# Combine and find the unique values
all_move_type = pd.concat([p1_move_type, p2_move_type]).unique()

print(all_move_type)

'''


"\n\n# Select both P1 and P2 move types, then explode them to break the lists apart\np1_move_type = df_flat_battle_timeline['p1_move_details.type'].explode()\np2_move_type = df_flat_battle_timeline['p2_move_details.type'].explode()\n\n# Combine and find the unique values\nall_move_type = pd.concat([p1_move_type, p2_move_type]).unique()\n\nprint(all_move_type)\n\n"

This tells us the move class:

- physical -> attack/defense
- special -> special attack/defense
- status -> (I don't know)
- nan -> (I suspect not doing anything/lost/switch pokemon)

In [57]:
'''


# Select both P1 and P2 move categories, then explode them to break the lists apart
p1_move_category = df_flat_battle_timeline['p1_move_details.category'].explode()
p2_move_category = df_flat_battle_timeline['p2_move_details.category'].explode()

# Combine and find the unique values
all_move_category = pd.concat([p1_move_category, p2_move_category]).unique()

print(all_move_category)

'''

"\n\n\n# Select both P1 and P2 move categories, then explode them to break the lists apart\np1_move_category = df_flat_battle_timeline['p1_move_details.category'].explode()\np2_move_category = df_flat_battle_timeline['p2_move_details.category'].explode()\n\n# Combine and find the unique values\nall_move_category = pd.concat([p1_move_category, p2_move_category]).unique()\n\nprint(all_move_category)\n\n"

In [60]:
import json
import pandas as pd
from pathlib import Path

train_file_path = 'data/train.jsonl'
test_file_path = 'data/test.jsonl'
sample_submission_path = 'data/sample_submission.csv'
train_data = []

# Read the file line by line
print(f"Loading data from '{train_file_path}'...")
try:
    with open(train_file_path, 'r') as f:
        for line in f:
            # json.loads() parses one line (one JSON object) into a Python dictionary
            train_data.append(json.loads(line))

    print(f"Successfully loaded {len(train_data)} battles.")

    # Let's inspect the first battle to see its structure
    print("\n--- Structure of the first train battle: ---")
    if train_data:
        first_battle = train_data[0]
        
        # To keep the output clean, we can create a copy and truncate the timeline
        battle_for_display = first_battle.copy()
        battle_for_display['battle_timeline'] = battle_for_display.get('battle_timeline', [])[:2] # Show first 2 turns
        
        # Use json.dumps for pretty-printing the dictionary
        print(json.dumps(battle_for_display, indent=4))
        if len(first_battle.get('battle_timeline', [])) > 3:
            print("    ...")
            print("    (battle_timeline has been truncated for display)")


except FileNotFoundError:
    print(f"ERROR: Could not find the training file at '{train_file_path}'.")
    print("Please make sure you have added the competition data to this notebook.")

Loading data from 'data/train.jsonl'...
Successfully loaded 10000 battles.

--- Structure of the first train battle: ---
{
    "battle_id": 0,
    "p1_team_details": [
        {
            "name": "tyranitar",
            "level": 100,
            "types": [
                "dark",
                "rock"
            ],
            "base_hp": 100,
            "base_atk": 134,
            "base_def": 110,
            "base_spa": 95,
            "base_spd": 100,
            "base_spe": 61,
            "item": "leftovers"
        },
        {
            "name": "typhlosion",
            "level": 100,
            "types": [
                "fire",
                "notype"
            ],
            "base_hp": 78,
            "base_atk": 84,
            "base_def": 78,
            "base_spa": 109,
            "base_spd": 85,
            "base_spe": 100,
            "item": "mintberry"
        },
        {
            "name": "golem",
            "level": 100,
            "types": [
       

### 2. Basic Feature Engineering

A successful model will likely require creating many complex features. For this starter notebook, however, we will create a very simple feature set based **only on the initial team stats**. This will be enough to train a model and generate a submission file.

It's up to you to engineer more powerful features!

In [76]:
import pandas as pd

type_chart = pd.read_csv("data/type_chart_gen2.csv")

type_chart['attacking_type'] = type_chart['attacking_type'].str.lower()
type_chart['defending_type'] = type_chart['defending_type'].str.lower()

display(type_chart.head())

def type_effectiveness(attacking_types, defending_types):
    if not attacking_types or not defending_types:
        return 1.0

    best_effectiveness = 0

    for attack_type in attacking_types:
        effectiveness = 1.0

        for defend_type in defending_types:
            match = type_chart[
                (type_chart['attacking_type'] == attack_type.lower()) &
                (type_chart['defending_type'] == defend_type.lower())
            ]

            # If combination isn't listed, it means normal effectiveness

            multiplier = 1.0 if match.empty else match['multiplier'].iloc[0]

            effectiveness *= multiplier

        best_effectiveness = max(best_effectiveness, effectiveness)

    return best_effectiveness
print(type_effectiveness(['fire'], ['water']))
print(type_effectiveness(['water'], ['fire']))

,attacking_type,defending_type,multiplier
0,normal,rock,0.5
1,normal,ghost,0.0
2,normal,steel,0.5
3,fire,fire,0.5
4,fire,water,0.5


0.5
2.0


In [77]:
from pyexpat import features
from tqdm.notebook import tqdm
import numpy as np




def create_simple_features(data):
    feature_list = []

    for battle in tqdm(data, desc="Extracting features"):
        features = {}

        # =========================================================
        # 1. PLAYER 1 TEAM FEATURES
        # =========================================================
        p1_team = battle.get('p1_team_details', [])

        if p1_team:
            stat_names = ['hp', 'spe', 'atk', 'def', 'spa', 'spd']

            for stat in stat_names:
                values = [p.get(f'base_{stat}', 0) for p in p1_team]

                features[f'p1_mean_{stat}'] = np.mean(values)
                features[f'p1_max_{stat}'] = np.max(values)
                features[f'p1_min_{stat}'] = np.min(values)
                features[f'p1_std_{stat}'] = np.std(values)

            # Range = strongest minus weakest stat on the team
            features['p1_hp_range'] = features['p1_max_hp'] - features['p1_min_hp']
            features['p1_atk_range'] = features['p1_max_atk'] - features['p1_min_atk']
            features['p1_spe_range'] = features['p1_max_spe'] - features['p1_min_spe']

            # Total overall stat strength of the whole team
            features['p1_total_stats'] = sum(
                p.get('base_hp', 0)
                + p.get('base_atk', 0)
                + p.get('base_def', 0)
                + p.get('base_spa', 0)
                + p.get('base_spd', 0)
                + p.get('base_spe', 0)
                for p in p1_team
            )

            # Number of different types represented on P1's team
            p1_types = set()
            for p in p1_team:
                for pokemon_type in p.get('types', []):
                    p1_types.add(pokemon_type)

            features['p1_unique_types'] = len(p1_types)

            # Number of different held items
            p1_items = [
                p.get('item')
                for p in p1_team
                if p.get('item')
            ]
            features['p1_unique_items'] = len(set(p1_items))


            # =====================================================
            # 2. PLAYER 1 LEAD FEATURES
            # =====================================================
            p1_lead = p1_team[0]

            features['p1_lead_hp'] = p1_lead.get('base_hp', 0)
            features['p1_lead_spe'] = p1_lead.get('base_spe', 0)
            features['p1_lead_atk'] = p1_lead.get('base_atk', 0)
            features['p1_lead_def'] = p1_lead.get('base_def', 0)
            features['p1_lead_spa'] = p1_lead.get('base_spa', 0)
            features['p1_lead_spd'] = p1_lead.get('base_spd', 0)


        # =========================================================
        # 3. PLAYER 2 LEAD FEATURES
        # =========================================================
        p2_lead = battle.get('p2_lead_details') or {}

        features['p2_lead_hp'] = p2_lead.get('base_hp', 0)
        features['p2_lead_spe'] = p2_lead.get('base_spe', 0)
        features['p2_lead_atk'] = p2_lead.get('base_atk', 0)
        features['p2_lead_def'] = p2_lead.get('base_def', 0)
        features['p2_lead_spa'] = p2_lead.get('base_spa', 0)
        features['p2_lead_spd'] = p2_lead.get('base_spd', 0)

        # =========================================================
        # 3.5 TYPE MATCHUP FEATURES
        # =========================================================

        if p1_team and p2_lead:
            p1_lead_types = p1_lead.get('types', [])
            p2_lead_types = p2_lead.get('types', [])

            # Lead vs lead effectiveness
            p1_lead_effectiveness = type_effectiveness(
                p1_lead_types,
                p2_lead_types
            )

            p2_lead_effectiveness = type_effectiveness(
                p2_lead_types,
                p1_lead_types
            )

            features['p1_lead_type_effectiveness'] = p1_lead_effectiveness
            features['p2_lead_type_effectiveness'] = p2_lead_effectiveness

            features['lead_type_advantage_diff'] = (
                p1_lead_effectiveness - p2_lead_effectiveness
            )

            # Whole P1 team vs P2 lead
            team_matchups = []

            for pokemon in p1_team:
                pokemon_types = pokemon.get('types', [])

                matchup = type_effectiveness(
                    pokemon_types,
                    p2_lead_types
                )

                team_matchups.append(matchup)

            features['p1_best_type_matchup'] = max(team_matchups)
            features['p1_mean_type_matchup'] = np.mean(team_matchups)

            features['p1_super_effective_count'] = sum(
                matchup > 1 for matchup in team_matchups
            )

            features['p1_resisted_count'] = sum(
                matchup < 1 for matchup in team_matchups
            )


        # =========================================================
        # 4. INITIAL LEAD MATCHUP
        # =========================================================
        if p1_team:
            features['lead_hp_diff'] = (
                features['p1_lead_hp'] - features['p2_lead_hp']
            )

            features['lead_speed_diff'] = (
                features['p1_lead_spe'] - features['p2_lead_spe']
            )

            features['lead_attack_diff'] = (
                features['p1_lead_atk'] - features['p2_lead_atk']
            )

            features['lead_defence_diff'] = (
                features['p1_lead_def'] - features['p2_lead_def']
            )

            features['lead_spa_diff'] = (
                features['p1_lead_spa'] - features['p2_lead_spa']
            )

            features['lead_spd_diff'] = (
                features['p1_lead_spd'] - features['p2_lead_spd']
            )


        # =========================================================
        # 5. TIMELINE FEATURES
        # =========================================================
        timeline = battle.get('battle_timeline', [])
        p1_move_types = set()
        p2_move_types = set()

        p1_move_categories = set()
        p2_move_categories = set()

        p1_first_switch_turn = 0
        p2_first_switch_turn = 0

        p1_first_faint_turn = 0
        p2_first_faint_turn = 0

        p1_seen = set()
        p2_seen = set()

        p1_fainted = set()
        p2_fainted = set()

        p1_hp_values = []
        p2_hp_values = []

        p1_status_turns = 0
        p2_status_turns = 0

        p1_effect_count = 0
        p2_effect_count = 0

        p1_switches = 0
        p2_switches = 0

        p1_physical_moves = 0
        p1_special_moves = 0
        p1_status_moves = 0

        p2_physical_moves = 0
        p2_special_moves = 0
        p2_status_moves = 0

        previous_p1 = None
        previous_p2 = None


        for turn_data in timeline:

            turn = turn_data.get('turn')

            p1_state = turn_data.get('p1_pokemon_state', {}) or {}
            p2_state = turn_data.get('p2_pokemon_state', {}) or {}

            p1_name = p1_state.get('name')
            p2_name = p2_state.get('name')

            p1_hp = p1_state.get('hp_pct', 0)
            p2_hp = p2_state.get('hp_pct', 0)


            # -----------------------------------------------------
            # Pokémon seen
            # -----------------------------------------------------
            if p1_name:
                p1_seen.add(p1_name)

            if p2_name:
                p2_seen.add(p2_name)


            # -----------------------------------------------------
            # HP values
            # -----------------------------------------------------
            p1_hp_values.append(p1_hp)
            p2_hp_values.append(p2_hp)


            # -----------------------------------------------------
            # Fainted Pokémon
            # -----------------------------------------------------
            if p1_name and p1_hp <= 0:
                p1_fainted.add(p1_name)

                if p1_first_faint_turn == 0:
                    p1_first_faint_turn = turn


            if p2_name and p2_hp <= 0:
                p2_fainted.add(p2_name)

                if p2_first_faint_turn == 0:
                    p2_first_faint_turn = turn

            # -----------------------------------------------------
            # Switches
            # -----------------------------------------------------
            if previous_p1 is not None and p1_name != previous_p1:
                p1_switches += 1
                
                if p1_first_switch_turn == 0:
                    p1_first_switch_turn = turn

            if previous_p2 is not None and p2_name != previous_p2:
                p2_switches += 1

                if p2_first_switch_turn == 0:
                    p2_first_switch_turn = turn

            previous_p1 = p1_name
            previous_p2 = p2_name


            # -----------------------------------------------------
            # Status conditions
            # -----------------------------------------------------
            p1_status = p1_state.get('status')
            p2_status = p2_state.get('status')

            if p1_status not in [None, 'nostatus', 'Unknown', 'NaN']:
                p1_status_turns += 1

            if p2_status not in [None, 'nostatus', 'Unknown', 'NaN']:
                p2_status_turns += 1


            # -----------------------------------------------------
            # Special effects
            # -----------------------------------------------------
            p1_effects = p1_state.get('effects', []) or []
            p2_effects = p2_state.get('effects', []) or []

            p1_effect_count += len(p1_effects)
            p2_effect_count += len(p2_effects)


            # -----------------------------------------------------
            # Move categories
            # -----------------------------------------------------
            p1_move = turn_data.get('p1_move_details', {}) or {}
            p2_move = turn_data.get('p2_move_details', {}) or {}

            p1_category = p1_move.get('category')
            p2_category = p2_move.get('category')

            if p1_category == 'physical':
                p1_physical_moves += 1
            elif p1_category == 'special':
                p1_special_moves += 1
            elif p1_category == 'status':
                p1_status_moves += 1

            if p2_category == 'physical':
                p2_physical_moves += 1
            elif p2_category == 'special':
                p2_special_moves += 1
            elif p2_category == 'status':
                p2_status_moves += 1

            p1_move = turn_data.get('p1_move_details', {}) or {}
            p2_move = turn_data.get('p2_move_details', {}) or {}

            p1_move_type = p1_move.get('type')
            p2_move_type = p2_move.get('type')

            p1_category = p1_move.get('category')
            p2_category = p2_move.get('category')

            if p1_move_type:
                p1_move_types.add(p1_move_type)

            if p2_move_type:
                p2_move_types.add(p2_move_type)

            if p1_category:
                p1_move_categories.add(p1_category)

            if p2_category:
                p2_move_categories.add(p2_category)
            # -----------------------------------------------------
            # Snapshots at turns 10, 20 and 30
            # -----------------------------------------------------
            if turn in [10, 20, 30]:

                features[f'p1_hp_turn_{turn}'] = p1_hp
                features[f'p2_hp_turn_{turn}'] = p2_hp

                features[f'hp_diff_turn_{turn}'] = (
                    p1_hp - p2_hp
                )
        # Unique move behaviour
        features['p1_unique_move_types'] = len(p1_move_types)
        features['p2_unique_move_types'] = len(p2_move_types)

        features['move_type_diff'] = (
            len(p1_move_types) - len(p2_move_types)
        )

        features['p1_unique_move_categories'] = len(p1_move_categories)
        features['p2_unique_move_categories'] = len(p2_move_categories)


        # Switch timing
        features['p1_first_switch_turn'] = p1_first_switch_turn
        features['p2_first_switch_turn'] = p2_first_switch_turn

        features['first_switch_diff'] = (
            p1_first_switch_turn - p2_first_switch_turn
        )


        # Faint timing
        features['p1_first_faint_turn'] = p1_first_faint_turn
        features['p2_first_faint_turn'] = p2_first_faint_turn

        features['first_faint_diff'] = (
            p2_first_faint_turn - p1_first_faint_turn
        )

        # =========================================================
        # 6. AGGREGATED TIMELINE FEATURES
        # =========================================================

        features['p1_pokemon_seen'] = len(p1_seen)
        features['p2_pokemon_seen'] = len(p2_seen)

        features['pokemon_seen_diff'] = (
            len(p1_seen) - len(p2_seen)
        )


        features['p1_fainted'] = len(p1_fainted)
        features['p2_fainted'] = len(p2_fainted)

        features['fainted_diff'] = (
            len(p2_fainted) - len(p1_fainted)
        )


        features['p1_switches'] = p1_switches
        features['p2_switches'] = p2_switches

        features['switch_diff'] = (
            p1_switches - p2_switches
        )


        features['p1_status_turns'] = p1_status_turns
        features['p2_status_turns'] = p2_status_turns

        features['status_turn_diff'] = (
            p2_status_turns - p1_status_turns
        )


        features['p1_effect_count'] = p1_effect_count
        features['p2_effect_count'] = p2_effect_count


        # Move usage
        features['p1_physical_moves'] = p1_physical_moves
        features['p1_special_moves'] = p1_special_moves
        features['p1_status_moves'] = p1_status_moves

        features['p2_physical_moves'] = p2_physical_moves
        features['p2_special_moves'] = p2_special_moves
        features['p2_status_moves'] = p2_status_moves


        # =========================================================
        # 7. HP SUMMARY FEATURES
        # =========================================================

        if p1_hp_values:
            features['p1_mean_battle_hp'] = np.mean(p1_hp_values)
            features['p1_min_battle_hp'] = np.min(p1_hp_values)
            features['p1_final_hp'] = p1_hp_values[-1]

        if p2_hp_values:
            features['p2_mean_battle_hp'] = np.mean(p2_hp_values)
            features['p2_min_battle_hp'] = np.min(p2_hp_values)
            features['p2_final_hp'] = p2_hp_values[-1]


        if p1_hp_values and p2_hp_values:

            features['mean_hp_diff'] = (
                np.mean(p1_hp_values)
                - np.mean(p2_hp_values)
            )

            features['final_hp_diff'] = (
                p1_hp_values[-1]
                - p2_hp_values[-1]
            )


        # =========================================================
        # 8. BATTLE MOMENTUM
        # =========================================================

        features['p1_hp_change_10_30'] = (
            features.get('p1_hp_turn_30', 0)
            - features.get('p1_hp_turn_10', 0)
        )

        features['p2_hp_change_10_30'] = (
            features.get('p2_hp_turn_30', 0)
            - features.get('p2_hp_turn_10', 0)
        )

        features['hp_advantage_change_10_20'] = (
            features.get('hp_diff_turn_20', 0)
            - features.get('hp_diff_turn_10', 0)
        )

        features['hp_advantage_change_20_30'] = (
            features.get('hp_diff_turn_30', 0)
            - features.get('hp_diff_turn_20', 0)
        )


        # =========================================================
        # ID + TARGET
        # =========================================================
        features['battle_id'] = battle.get('battle_id')

        if 'player_won' in battle:
            features['player_won'] = int(battle['player_won'])

        feature_list.append(features)


    return pd.DataFrame(feature_list).fillna(0)

# Create feature DataFrames for both training and test sets
print("Processing training data...")
train_df = create_simple_features(train_data)

print("\nProcessing test data...")
test_data = []
with open(test_file_path, 'r') as f:
    for line in f:
        test_data.append(json.loads(line))
test_df = create_simple_features(test_data)

print("\nTraining features preview:")
display(train_df.head())

Processing training data...


Extracting features:   0%|          | 0/10000 [00:00<?, ?it/s]


Processing test data...


Extracting features:   0%|          | 0/5000 [00:00<?, ?it/s]


Training features preview:


,p1_mean_hp,p1_max_hp,p1_min_hp,p1_std_hp,p1_mean_spe,p1_max_spe,p1_min_spe,p1_std_spe,p1_mean_atk,p1_max_atk,...,p2_min_battle_hp,p2_final_hp,mean_hp_diff,final_hp_diff,p1_hp_change_10_30,p2_hp_change_10_30,hp_advantage_change_10_20,hp_advantage_change_20_30,battle_id,player_won
0,83.000000,100,55,14.719601,84.333333,120,45,28.645341,88.000000,134,...,0.26,0.850000,-0.102000,0.150000,0.00000,-0.150000,-1.000000,1.150000,0,0
1,91.666667,160,60,33.499585,70.833333,115,30,28.783193,83.333333,110,...,0.56,0.740000,-0.101333,-0.740000,-0.91000,-0.200000,-0.180000,-0.530000,1,0
2,95.000000,160,50,33.040379,69.333333,115,30,26.310116,113.166667,134,...,0.00,0.937337,-0.063423,-0.749347,0.18799,-0.041775,1.169713,-0.939948,2,0
3,119.166667,255,50,69.846061,65.000000,100,30,20.816660,75.000000,110,...,0.40,0.950000,-0.048000,-0.410000,-0.46000,0.030000,-0.260000,-0.230000,3,0
4,105.166667,160,75,30.140873,60.833333,100,30,26.047499,89.500000,110,...,0.00,0.000000,0.064333,0.000000,-1.00000,-1.000000,-0.430000,0.430000,4,0


### 3. Training a Baseline Model

Now that we have some features, let's train a simple `LogisticRegression` model. This will give us a starting point for our predictions.

In [78]:
#Now we make a validation set so we can evaluate our model before making predictions on the test set. We will use 80% of the training data for training and 20% for validation.
from sklearn.model_selection import train_test_split

features = [
    col for col in train_df.columns
    if col not in ['battle_id', 'player_won']
]

X = train_df[features]
y = train_df['player_won']

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

Training: (8000, 107)
Validation: (2000, 107)


In [79]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
# Define our features (X) and target (y)
features = [col for col in train_df.columns if col not in ['battle_id', 'player_won']]
#X_train = train_df[features]
#y_train = train_df['player_won']


# Align columns - crucial for ensuring test set has same features as train set
X_test = test_df[features]

# Initialize and train the model
print("Training a simple Logistic Regression model...")
model = LogisticRegression(random_state=42, max_iter=10000) # Remember to set random state!!
model.fit(X_train, y_train)
print("Model training complete.")

y_val_pred = model.predict(X_val)
validation_accuracy = accuracy_score(y_val, y_val_pred)

print("Validation accuracy:", validation_accuracy)

Training a simple Logistic Regression model...
Model training complete.
Validation accuracy: 0.6365


/opt/anaconda3/envs/pokemon/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [80]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

scaled_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=5000)
)

scaled_model.fit(X_train, y_train)

scaled_pred = scaled_model.predict(X_val)

scaled_accuracy = accuracy_score(y_val, scaled_pred)

print("Scaled Logistic Regression accuracy:", scaled_accuracy)

Scaled Logistic Regression accuracy: 0.6405


In [81]:
#Now using a ranfom forest classifier to see if we can improve our accuracy
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_val_pred = rf_model.predict(X_val)

rf_accuracy = accuracy_score(y_val, rf_val_pred)

print("Random Forest validation accuracy:", rf_accuracy)

Random Forest validation accuracy: 0.6355


In [82]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)

lgbm_model.fit(X_train, y_train)

lgbm_pred = lgbm_model.predict(X_val)

lgbm_accuracy = accuracy_score(y_val, lgbm_pred)

print("LightGBM validation accuracy:", lgbm_accuracy)

[LightGBM] [Info] Number of positive: 4000, number of negative: 4000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008070 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10366
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 105
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
LightGBM validation accuracy: 0.6495


In [83]:
import pandas as pd

importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': lgbm_model.feature_importances_
}).sort_values('importance', ascending=False)

display(importance_df.head(30))

,feature,importance
98,p2_mean_battle_hp,405
101,mean_hp_diff,349
95,p1_mean_battle_hp,342
94,p2_status_moves,315
12,p1_mean_def,307
62,p2_hp_turn_30,301
19,p1_std_spa,301
99,p2_min_battle_hp,292
7,p1_std_spe,291
23,p1_std_spd,288


In [84]:
top_features = importance_df.head(30)['feature'].tolist()

X_train_top = X_train[top_features]
X_val_top = X_val[top_features]

lgbm_top = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)

lgbm_top.fit(X_train_top, y_train)

pred = lgbm_top.predict(X_val_top)

print("Top 30 feature accuracy:",
      accuracy_score(y_val, pred))

[LightGBM] [Info] Number of positive: 4000, number of negative: 4000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001175 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6751
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Top 30 feature accuracy: 0.593


In [85]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000, random_state=42)
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print("Fold accuracies:", scores)
print("Mean accuracy:", scores.mean())

Fold accuracies: [0.637  0.6495 0.641  0.645  0.629 ]
Mean accuracy: 0.6403


### 4. Creating the Submission File

The competition requires a `.csv` file with two columns: `battle_id` and `player_won`. Let's use our trained model to make predictions on the test set and format them correctly.

In [86]:
# Make predictions on the test data
print("Generating predictions on the test set...")
test_predictions = lgbm_model.predict(X_test)

# Use the provided sample file as the submission template.
sample_submission = pd.read_csv(sample_submission_path)
expected_columns = ['battle_id', 'player_won']
if list(sample_submission.columns) != expected_columns:
    raise ValueError(f"Expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

# Match predictions to the template by battle_id, preserving its row order.
predictions_by_id = pd.Series(test_predictions, index=test_df['battle_id'])
submission_df = sample_submission.copy()
submission_df['player_won'] = submission_df['battle_id'].map(predictions_by_id)
if submission_df['player_won'].isna().any():
    raise ValueError("Some sample submission battle_id values have no matching test prediction.")
submission_df['player_won'] = submission_df['player_won'].astype(int)

submission_df.to_csv('submission.csv', index=False)

# Once you create the submission file, you will find it in /kaggle/working
print("\n'submission.csv' file created successfully!")
display(submission_df.head())

Generating predictions on the test set...

'submission.csv' file created successfully!


,battle_id,player_won
0,0,1
1,1,0
2,2,0
3,3,1
4,4,0


### 5. Submitting Your Results

Once you have generated your `submission.csv` file, there are two primary ways to submit it to the competition.

---

#### Method A: Submitting Directly from the Notebook (Recommended)

This is the standard method for code competitions. It ensures that your submission is linked to the code that produced it, which is crucial for reproducibility.

1.  **Save Your Work:** Click the **"Save Version"** button in the top-right corner of the notebook editor.
2.  **Run the Notebook:** In the pop-up window, select **"Save & Run All (Commit)"** and then click the **"Save"** button. This will run your entire notebook from top to bottom and save the output, including your `submission.csv` file.
3.  **Go to the Viewer:** Once the save process is complete, navigate to the notebook viewer page. 
4.  **Submit to Competition:** In the viewer, find the **"Submit to Competition"** section. This is usually located in the header of the output section or in the vertical "..." menu on the right side of the page. Clicking the **Submit** button this will submit your generated `submission.csv` file.

After submitting, you will see your score in the **"Submit to Competition"** section or in the [Public Leaderboard](https://www.kaggle.com/competitions/fds-pokemon-battles-prediction-2026/leaderboard?).

---

#### Method B: Manual Upload

You can also generate your predictions and submission file using any environment you prefer (this notebook, Google Colab, or your local machine).

1.  **Generate the `submission.csv` file** using your model.
2.  **Download the file** to your computer.
3.  **Navigate to the [Leaderboard Page](https://www.kaggle.com/competitions/fds-pokemon-battles-prediction-2026/leaderboard?)** and click on the **"Submit Predictions"** button.
4.  **Upload Your File:** Drag and drop or select your `submission.csv` file to upload it.

This method is quick, but keep in mind that for the final evaluation, you might be required to provide the code that generated your submission.

Good luck!